In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
print(train.shape)

(2000, 8)


In [4]:
print(train["answer"].value_counts())
print(eval("490+324"))

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
814


In [5]:
import string
# prompts = train["prompt"].as_type(str).str.lower()
# table
string1 = ""
for i in train["prompt"]:
    string1 += i.lower()

translator = str.maketrans("", "", string.punctuation)
no_punct = string1.translate(translator)

words = no_punct.split()
vocab = set(word for word in words)
vocab
print(len(vocab))

1318


In [6]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
row1 = train["prompt"][0].lower()
row1_cleaned = row1.translate(translator)
row1_cleaned = row1_cleaned.split()
print(row1_cleaned)
# print(ENGLISH_STOP_WORDS)
filtered = [word for word in row1_cleaned if word not in ENGLISH_STOP_WORDS]
print(filtered)
print(len(filtered))

['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
13


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
combined_text = (
    train["prompt"].astype(str) + " " +
    train["A"].astype(str) + " " +
    train["B"].astype(str) + " " +
    train["C"].astype(str) + " " +
    train["D"].astype(str) + " " +
    train["E"].astype(str)
).tolist()

# print(combined_text[0])
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(combined_text)
print(X.shape)

(2000, 2762)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
row1 = train.loc[0]
prompt_1 = str(row1["prompt"])
optionA_1 = str(row1["A"])
tfidf_pair = vectorizer.transform([prompt_1, optionA_1])

sim_matrix = cosine_similarity(tfidf_pair[0], tfidf_pair[1])
similarity = sim_matrix[0, 0]

similarity_rounded = round(similarity, 4)
print(similarity_rounded)

0.272


In [9]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

option_cols = ["A", "B", "C", "D", "E"]

correct = 0
total = len(train)

for idx, row in train.iterrows():
    prompt_text = str(row["prompt"])
    options_text = [str(row[col]) for col in option_cols]

    
    texts = [prompt_text] + options_text  
    tfidf = vectorizer.transform(texts)   

    prompt_vec = tfidf[0]        
    options_vec = tfidf[1:]     

    
    sims = cosine_similarity(prompt_vec, options_vec)[0] 

    
    best_idx = sims.argmax()
    predicted_option = option_cols[best_idx]

    
    if predicted_option == row["answer"]:
        correct += 1

accuracy = correct / total * 100.0
print(round(accuracy, 2), "%")

13.55 %


In [10]:
top3 = ['B','C','A']
def map3(preds,correct):
    score = 0
    for i in range(3):
        if preds[i] == correct:
            score = 1/(1+i)
            break
    return score
map3_sum = 0
for answer in train["answer"]:
    map3_sum += map3(top3,answer)
map3_sum/2000

0.4212500000000017

In [11]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

option_cols = ["A", "B", "C", "D", "E"]

def ap3(true_label, preds):
    score = 0.0
    for k in range(3):
        if preds[k] == true_label:
            score = 1.0 / (k + 1)
            break
    return score

ap_scores = []

for _, row in train.iterrows():
    prompt_text = str(row["prompt"])
    options_text = [str(row[col]) for col in option_cols]

    texts = [prompt_text] + options_text
    tfidf = vectorizer.transform(texts)    

    prompt_vec = tfidf[0]
    options_vec = tfidf[1:]              

    sims = cosine_similarity(prompt_vec, options_vec)[0] 

    order = np.argsort(-sims)  
    top3_indices = order[:3]
    top3_labels = [option_cols[i] for i in top3_indices]

    ap_scores.append(ap3(row["answer"], top3_labels))

map3 = sum(ap_scores) / len(ap_scores)
print(round(map3, 4))

0.2915
